# LLaMA 3.1 8B vs LLaMA 3.3 70B

Same prompt, decoding and truncation with both models on the 2,000 Inspec abstracts and on the 182-article validation sample (Section IV-A4): agreement between the two keyword sets per document, and both models against the Inspec gold keyphrases with the paired difference and its 95% CI. No new LLM call.

Inputs: 8B and 70B outputs for Inspec and for the sample (repository root and `data/`), `dataset_inspec.csv`. Outputs in `results/e2_model_agreement/`. Gate: the 8B Inspec means must match notebook 6 within 5e-5.

In [1]:
# ============================================================
# CONFIGURATION AND IMPORTS
# ============================================================
import os
os.environ["OMP_NUM_THREADS"] = "4"          # several notebooks run concurrently on one machine
import sys
sys.path.insert(0, "scripts")
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import common as C
import inspec_evaluation as IE

torch.set_num_threads(4)
C.set_seeds()                                  # Python, NumPy and Torch seeds (42), deterministic algorithms
T = C.Timer()

OUT = Path("results/e2_model_agreement")
OUT.mkdir(parents=True, exist_ok=True)


def public_or_private(name: str) -> Path:
    """Public copy in data/ first; fall back to the private directory (FTTS_PRIVATE_DIR) when it is missing."""
    p = Path("data") / name
    return p if p.exists() else C.PRIVATE_DIR / name


PATHS = {
    "inspec":     Path("dataset_inspec.csv"),                        # 2,000 Inspec abstracts with gold keyphrases
    "inspec_8b":  Path("inspec_llama-3.1-8b-EN.csv"),                # published LLaMA 3.1 8B outputs on Inspec
    "inspec_70b": public_or_private("inspec_llama-3.3-70b-EN.csv"),  # LLaMA 3.3 70B outputs on Inspec
    "sample_8b":  Path("data/keywords_llm_llama-3.1-8b-EN.csv"),     # 8B outputs on the 182-article validation sample
    "sample_70b": public_or_private("keywords_llm_llama-3.3-70b-EN.csv"),  # 70B outputs on the same sample
}
inputs = {}
for k, p in PATHS.items():
    inputs[k] = {"path": str(p), "present": p.exists(), "sha256": C.sha256(p) if p.exists() else None,
                 "redistributed": str(C.REPO_ROOT) in str(p.resolve())}
print("=== Inputs ===")
print(pd.DataFrame(inputs).T.to_string())

meta = C.env_metadata(experiment="E2 model agreement", inputs=inputs, llm_calls=0, paid_api_calls=0)
print("\n=== Environment ===")
print(json.dumps({k: meta[k] for k in ("python", "platform", "machine", "cpu_count", "git_commit",
                                       "embedding_model", "embedding_model_revision")}, indent=2))
print("packages:", json.dumps(meta["packages"]))

print(f"\nLoading embedding model: {C.MODEL_NAME} (revision {C.MODEL_REVISION[:12]}, CPU, 4 threads)")
model = C.load_embedder(threads=4)
summary, validation = {}, {}

=== Inputs ===
                                              path present                                                            sha256 redistributed
inspec                          dataset_inspec.csv    True  0a84eb5ca79c418987a47f391a6bbf47e98a6ad4ac8fcc9a94a0ae348dc5765d          True
inspec_8b               inspec_llama-3.1-8b-EN.csv    True  18d254befe74d016630afcb580a7e838cda7dd5d76fb6af85a4d1030d9c1f781          True
inspec_70b        data/inspec_llama-3.3-70b-EN.csv    True  ca03fe568a7ee2eb113b9fa5b9f3160e73f8acf24be6036b142eeb407cb98a1c          True
sample_8b    data/keywords_llm_llama-3.1-8b-EN.csv    True  12dc39067c1d4775725bc1111d4ef859df843c2166b95650ea48211eb125f2e0          True
sample_70b  data/keywords_llm_llama-3.3-70b-EN.csv    True  db7490aa55d0e1556af6515658a59c00d1dfa6c125883f821bfce86c08cc60b2          True


/home/mat/academic-writing/papers/from_text_to_structure/data_repo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



=== Environment ===
{
  "python": "3.12.3",
  "platform": "Linux-7.0.0-31-generic-x86_64-with-glibc2.39",
  "machine": "x86_64",
  "cpu_count": 16,
  "git_commit": "af664fad8d65723998851025c9d052e9bf1fa73e",
  "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
  "embedding_model_revision": "e8f8c211226b894fcb81acc59f3b34ba3efd5f42"
}
packages: {"numpy": "2.1.3", "pandas": "2.2.3", "scipy": "1.15.3", "sklearn": "1.9.0", "networkx": "3.6.1", "community": "0.16", "sentence_transformers": "5.1.1", "torch": "2.8.0+cpu", "transformers": "4.57.6", "langdetect": "unknown", "openai": "1.107.2"}

Loading embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 (revision e8f8c211226b, CPU, 4 threads)


In [2]:
# ============================================================
# METRIC HELPERS (notebook 6 functions via inspec_evaluation; embeddings cached per phrase)
# ============================================================
def make_embed_fn(model):
    cache = {}

    def embed(phrases):
        missing = [p for p in phrases if p not in cache]
        if missing:
            vecs = model.encode(missing, batch_size=IE.BATCH_SIZE, show_progress_bar=False,
                                normalize_embeddings=False, convert_to_numpy=True)
            cache.update(zip(missing, vecs))
        return np.vstack([cache[p] for p in phrases])
    return embed


def score_pair(ref, pred, embed):
    """Six notebook-6 metrics of ``pred`` against ``ref`` (both cleaned lists), plus set identity."""
    m = IE.soft_matching_metrics(ref, pred, embed, IE.TAU)
    m["jaccard_lex"] = IE.jaccard(ref, pred)
    m["global_sem_sim"] = IE.global_concat_similarity(ref, pred, embed)
    m["identical_set"] = float(set(ref) == set(pred))
    return m


def agreement_table(list_a, list_b, embed, ids, label_a, label_b):
    """Per-document agreement of two keyword lists (a = reference side, b = prediction side)."""
    rows = []
    for i, a, b in zip(ids, list_a, list_b):
        ca, cb = IE.clean_list(a), IE.clean_list(b)
        m = score_pair(ca, cb, embed)
        rows.append({"doc": i, f"n_{label_a}": len(ca), f"n_{label_b}": len(cb), **m})
    df = pd.DataFrame(rows)
    summ = {k: {"mean": float(df[k].mean()), "sd": float(df[k].std(ddof=1))} for k in IE.METRICS}
    summ["identical_set_share"] = float(df.identical_set.mean())
    summ["documents"] = int(len(df))
    return df, summ


def agreement_frame(summ: dict) -> pd.DataFrame:
    """Mean and SD of the six metrics as a printable table."""
    return pd.DataFrame({k: v for k, v in summ.items() if isinstance(v, dict)}).T


embed = make_embed_fn(model)
print("tau =", IE.TAU, "| metrics:", IE.METRICS)

tau = 0.7 | metrics: ['jaccard_lex', 'soft_precision', 'soft_recall', 'soft_f1', 'soft_mean_max', 'global_sem_sim']


In [3]:
# ============================================================
# INSPEC: REPRODUCE THE PUBLISHED 8B SCORES (gate, tolerance 5e-5 on the four-decimal means)
# ============================================================
insp = pd.read_csv(PATHS["inspec"])
gold = [IE.clean_list(IE.safe_parse_list(x)) for x in insp["keywords_gt"]]
p8 = pd.read_csv(PATHS["inspec_8b"]).sort_values("row_abs")
pred8 = [IE.clean_list(IE.safe_parse_list(x)) for x in p8["keywords_llm"]]
print(f"Inspec documents: {len(insp)}  |  8B prediction rows: {len(p8)}")

rows = [{"row_abs": r, **score_pair(g, p, embed)} for r, g, p in zip(p8.row_abs, gold, pred8)]
g8 = pd.DataFrame(rows)
means8 = {k: round(float(g8[k].mean()), 4) for k in IE.METRICS}
gate = {k: {"observed": means8[k], "reference": IE.EXPECTED[k], "pass": abs(means8[k] - IE.EXPECTED[k]) <= 5e-5} for k in IE.METRICS}
gate_pass = all(v["pass"] for v in gate.values())
validation["inspec_8b_reproduction"] = {"pass": gate_pass, "checks": gate}
print("\nINSPEC 8B GATE:", "PASS" if gate_pass else "FAIL")
print(pd.DataFrame(gate).T.to_string())
gold_table = [{"model": "llama-3.1-8b-instruct", **{k: float(g8[k].mean()) for k in IE.METRICS},
               **{f"{k}_sd": float(g8[k].std(ddof=1)) for k in IE.METRICS}}]
T.mark("inspec_8b")
print(f"\ninspec_8b: {T.marks['inspec_8b']} s")
if not gate_pass:
    raise RuntimeError("The published 8B Inspec means were not reproduced; stopping before the 70B comparison.")

Inspec documents: 2000  |  8B prediction rows: 2000



INSPEC 8B GATE: PASS
               observed reference  pass
jaccard_lex      0.1422    0.1422  True
soft_precision   0.7939    0.7939  True
soft_recall       0.467     0.467  True
soft_f1          0.5652    0.5652  True
soft_mean_max    0.7522    0.7522  True
global_sem_sim   0.8042    0.8042  True

inspec_8b: 2267.383 s


In [4]:
# ============================================================
# INSPEC: 70B AGAINST THE GOLD KEYPHRASES, 8B vs 70B AGREEMENT, PAIRED DIFFERENCE
# ============================================================
if inputs["inspec_70b"]["present"]:
    p70 = pd.read_csv(PATHS["inspec_70b"]).sort_values("row_abs")
    assert list(p70.row_abs) == list(p8.row_abs)
    pred70 = [IE.clean_list(IE.safe_parse_list(x)) for x in p70["keywords_llm"]]
    g70 = pd.DataFrame([{"row_abs": r, **score_pair(g, p, embed)} for r, g, p in zip(p70.row_abs, gold, pred70)])
    gold_table.append({"model": "llama-3.3-70b-instruct", **{k: float(g70[k].mean()) for k in IE.METRICS},
                       **{f"{k}_sd": float(g70[k].std(ddof=1)) for k in IE.METRICS}})
    agree_df, agree = agreement_table(pred8, pred70, embed, p8.row_abs, "8b", "70b")
    agree_df.to_csv(OUT / "inspec_agreement_8b_vs_70b_per_document.csv", index=False)
    # Paired difference of gold metrics between models (per document), for a CI without new runs.
    diff = {k: {"mean_70b_minus_8b": float((g70[k] - g8[k]).mean()),
                "ci95_low": float((g70[k] - g8[k]).mean() - 1.96 * (g70[k] - g8[k]).std(ddof=1) / np.sqrt(len(g8))),
                "ci95_high": float((g70[k] - g8[k]).mean() + 1.96 * (g70[k] - g8[k]).std(ddof=1) / np.sqrt(len(g8)))}
            for k in IE.METRICS}
    summary["inspec"] = {"documents": int(len(g8)), "agreement_8b_vs_70b": agree,
                         "gold_metrics_paired_difference": diff,
                         "empty_predictions": {"8b": int(sum(len(p) == 0 for p in pred8)), "70b": int(sum(len(p) == 0 for p in pred70))}}
    print("=== Inspec: each model against the gold keyphrases (means over 2,000 documents) ===")
    print(pd.DataFrame(gold_table).set_index("model")[IE.METRICS].T.to_string())
    print("\n=== Inspec: paired per-document difference, 70B minus 8B (normal 95% CI) ===")
    print(pd.DataFrame(diff).T.to_string())
    print("\n=== Inspec: agreement between the 8B and 70B keyword sets (8B as reference side) ===")
    print(agreement_frame(agree).to_string())
    print(f"identical keyword sets: {agree['identical_set_share']:.4f} of {agree['documents']} documents")
    print(f"empty predictions: 8B {summary['inspec']['empty_predictions']['8b']}, 70B {summary['inspec']['empty_predictions']['70b']}")
else:
    summary["inspec"] = {"documents": int(len(g8)), "agreement_8b_vs_70b": "70B prediction file not available"}
    print("70B Inspec prediction file not available; agreement skipped.")
pd.DataFrame(gold_table).to_csv(OUT / "inspec_gold_metrics_by_model.csv", index=False)
T.mark("inspec_70b")
print(f"\ninspec_70b: {T.marks['inspec_70b'] - T.marks['inspec_8b']:.1f} s")

=== Inspec: each model against the gold keyphrases (means over 2,000 documents) ===
model           llama-3.1-8b-instruct  llama-3.3-70b-instruct
jaccard_lex                  0.142228                0.132218
soft_precision               0.793900                0.814050
soft_recall                  0.466999                0.478940
soft_f1                      0.565183                0.580934
soft_mean_max                0.752155                0.756959
global_sem_sim               0.804198                0.811026

=== Inspec: paired per-document difference, 70B minus 8B (normal 95% CI) ===
                mean_70b_minus_8b  ci95_low  ci95_high
jaccard_lex             -0.010009 -0.013889  -0.006130
soft_precision           0.020150  0.012514   0.027786
soft_recall              0.011941  0.006916   0.016965
soft_f1                  0.015751  0.010210   0.021293
soft_mean_max            0.004804  0.001813   0.007796
global_sem_sim           0.006828  0.003821   0.009835

=== Inspec: agreem

In [5]:
# ============================================================
# MATHEMATICS-EDUCATION VALIDATION SAMPLE: 8B vs 70B AGREEMENT
# ============================================================
s8 = pd.read_csv(PATHS["sample_8b"]).sort_values("row_abs")
human = pd.read_csv("human_eval_M1_8b.csv")
print(f"Validation sample rows in the 8B file: {len(s8)}  |  articles covered by the human evaluation "
      f"(7. HUMAN_EVAL_M1.ipynb): {human.id_inv.nunique()}")
print(f"All {len(s8)} rows enter the agreement measurement; no row is dropped.")
if inputs["sample_70b"]["present"]:
    s70 = pd.read_csv(PATHS["sample_70b"]).sort_values("row_abs")
    assert list(s70.row_abs) == list(s8.row_abs)
    a8 = [IE.safe_parse_list(x) for x in s8.keywords_llm]
    a70 = [IE.safe_parse_list(x) for x in s70.keywords_llm]
    sdf, sagree = agreement_table(a8, a70, embed, s8.row_abs, "8b", "70b")
    sdf.to_csv(OUT / "matheduc_sample_agreement_8b_vs_70b_per_document.csv", index=False)
    summary["mathematics_education_sample"] = {"documents": int(len(sdf)), "agreement_8b_vs_70b": sagree}
    print("\n=== Validation sample: agreement between the 8B and 70B keyword sets ===")
    print(agreement_frame(sagree).to_string())
    print(f"identical keyword sets: {sagree['identical_set_share']:.4f} of {sagree['documents']} documents")
    print("\nFirst rows of the per-document table:")
    print(sdf.head(5).to_string(index=False))
else:
    summary["mathematics_education_sample"] = {"documents": int(len(s8)), "agreement_8b_vs_70b": "70B prediction file not available"}
    print("70B sample prediction file not available; agreement skipped.")
T.mark("sample")
print(f"\nsample: {T.marks['sample'] - T.marks['inspec_70b']:.1f} s")

Validation sample rows in the 8B file: 182  |  articles covered by the human evaluation (7. HUMAN_EVAL_M1.ipynb): 179
All 182 rows enter the agreement measurement; no row is dropped.



=== Validation sample: agreement between the 8B and 70B keyword sets ===
                    mean        sd
jaccard_lex     0.440215  0.238442
soft_precision  0.783516  0.180707
soft_recall     0.795604  0.183226
soft_f1         0.785464  0.175011
soft_mean_max   0.854135  0.133626
global_sem_sim  0.905556  0.130935
identical keyword sets: 0.0495 of 182 documents

First rows of the per-document table:
 doc  n_8b  n_70b  soft_recall  soft_precision  soft_f1  soft_mean_max  jaccard_lex  global_sem_sim  identical_set
   0     5      5          0.8             0.8 0.800000       0.919996     0.666667        0.887131            0.0
   1     5      5          0.8             0.8 0.800000       0.880792     0.666667        0.921971            0.0
   2     5      5          1.0             0.8 0.888889       0.927708     0.666667        0.952799            0.0
   3     5      5          1.0             1.0 1.000000       0.980801     0.666667        0.981744            0.0
   4     5      5  

In [6]:
# ============================================================
# WRITE SUMMARY, VALIDATION AND METADATA
# ============================================================
C.write_json(summary, OUT / "summary.json")
C.write_json(validation, OUT / "validation.json")
meta.update({"completed_utc": C.now_utc(), "timings_seconds": T.marks, "status": "complete", "gate_pass": gate_pass})
C.write_json(meta, OUT / "metadata.json")
print("timings (s):", T.marks, "| gate_pass:", gate_pass)
print("files written to", OUT)
for name in ["inspec_gold_metrics_by_model.csv", "inspec_agreement_8b_vs_70b_per_document.csv",
             "matheduc_sample_agreement_8b_vs_70b_per_document.csv", "summary.json", "validation.json", "metadata.json"]:
    print(f"  {name:55s} {(OUT / name).stat().st_size:>10,} bytes")
print("\n=== summary.json ===")
print(json.dumps(summary, indent=2))

timings (s): {'inspec_8b': 2267.383, 'inspec_70b': 2640.144, 'sample': 2661.342} | gate_pass: True
files written to results/e2_model_agreement
  inspec_gold_metrics_by_model.csv                               665 bytes
  inspec_agreement_8b_vs_70b_per_document.csv                169,251 bytes
  matheduc_sample_agreement_8b_vs_70b_per_document.csv        15,607 bytes
  summary.json                                                 2,658 bytes
  validation.json                                                738 bytes
  metadata.json                                                2,057 bytes

=== summary.json ===
{
  "inspec": {
    "documents": 2000,
    "agreement_8b_vs_70b": {
      "jaccard_lex": {
        "mean": 0.39507857142857145,
        "sd": 0.25213350996705997
      },
      "soft_precision": {
        "mean": 0.7651500000000001,
        "sd": 0.18672951415276554
      },
      "soft_recall": {
        "mean": 0.7557166666666667,
        "sd": 0.18607844547854982
      },
      "